In [26]:
# install packages into this notebook's .venv (uv envs have no pip, so use %uv)
%uv pip install langgraph google-genai python-dotenv

/Users/jinyuan/Documents/ClaudeProject/AAI520-Group-7/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [27]:
from google import genai
from google.genai import types
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()   # reads GEMINI_API_KEY from .env
MODEL = "gemini-3.8-flash"

# 1. Tools declaration

In [31]:
from datetime import datetime, timedelta, timezone

# ---------------------------------------------------------------------------
# MOCK news tool -- placeholder until the real data tool is ready.
#
# Contract the real tool must follow (same name, args, and return shape):
#   get_news(ticker: str, limit: int = 10) -> list[dict]
#   each dict has:
#     "title":        str   headline
#     "source":       str   publisher, e.g. "Reuters"
#     "published_at": str   ISO 8601 UTC timestamp
#     "url":          str   link to the article
#     "text":         str   article body or summary
# ---------------------------------------------------------------------------
def get_news(ticker: str, limit: int = 10) -> list[dict]:
    now = datetime.now(timezone.utc)
    templates = [
        ("{t} beats Q3 earnings estimates as revenue rises 8% year over year", "Reuters",
         "{t} reported quarterly EPS above analyst consensus, driven by strong services growth. "
         "Management raised full-year guidance."),
        ("{t} announces new product line at annual developer event", "Bloomberg",
         "{t} unveiled several new products and a partnership aimed at expanding its ecosystem."),
        ("Fed holds rates steady; tech stocks including {t} edge higher", "CNBC",
         "The Federal Reserve left interest rates unchanged. Large-cap tech, including {t}, "
         "gained on expectations of cuts next year."),
        ("Analyst downgrades {t} to Hold, citing valuation concerns", "MarketWatch",
         "A major brokerage cut its rating on {t}, saying the stock already prices in most "
         "near-term growth."),
        ("{t} faces regulatory scrutiny over app store practices in the EU", "Financial Times",
         "EU regulators opened an inquiry into {t}'s business practices, which could lead to fines."),
        ("{t} supplier reports production delays in Asia", "Nikkei Asia",
         "A key component supplier to {t} warned of shipment delays this quarter."),
    ]
    items = []
    for i, (title, source, text) in enumerate(templates):
        items.append({
            "title": title.format(t=ticker),
            "source": source,
            "published_at": (now - timedelta(hours=12 * i)).isoformat(timespec="seconds"),
            "url": f"https://example.com/{ticker.lower()}/news/{i}",
            "text": text.format(t=ticker),
        })
    return items[:limit]


# quick check
get_news("AAPL", limit=2)

[{'title': 'AAPL beats Q3 earnings estimates as revenue rises 8% year over year',
  'source': 'Reuters',
  'published_at': '2026-10-08T15:25:02+00:00',
  'url': 'https://example.com/aapl/news/0',
  'text': 'AAPL reported quarterly EPS above analyst consensus, driven by strong services growth. Management raised full-year guidance.'},
 {'title': 'AAPL announces new product line at annual developer event',
  'source': 'Bloomberg',
  'published_at': '2026-10-08T03:25:02+00:00',
  'url': 'https://example.com/aapl/news/1',
  'text': 'AAPL unveiled several new products and a partnership aimed at expanding its ecosystem.'}]

# 2. Agent Declaration

In [46]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

# create state for multi-step workflow
class State(TypedDict):
    ticker: str
    plan: List[str]
    raw_news: list
    routed: dict
    analysis: str
    critique: str
    score: float
    iterations: int

class Plan(BaseModel):
    steps: List[str]

# Entry point
def planner(state: State) -> dict:
    ticker = state["ticker"].upper()
    response = client.models.generate_content(
        model=MODEL,
        contents=(
            f"Create a 4-6 step research plan to analyze the stock {ticker}. "
            "Cover data gathering (news, earnings, price), analysis, and self-critique. "
            "Keep each step short and actionable."
        ),
        config=types.GenerateContentConfig(
            system_instruction="You are an investment research agent that plans before acting.",
            response_mime_type="application/json",
            response_schema=Plan,
        ),
    )
    if response.parsed is None:
        raise RuntimeError(f"Planner returned no valid plan: {response.text}")

    return {"ticker": ticker, "plan": response.parsed.steps, "iterations": 0}

# Step 2: gather raw data with the news tool (mock for now)
def fetch(state: State) -> dict:
    ticker = state["ticker"]
    try:
        items = get_news(ticker, limit=10)
    except Exception as e:
        print(f"[fetch] news tool failed for {ticker}: {e}")
        items = []

    # drop duplicates by title, newest first
    seen, raw_news = set(), []
    for item in sorted(items, key=lambda x: x["published_at"], reverse=True):
        if item["title"] not in seen:
            seen.add(item["title"])
            raw_news.append(item)

    print(f"[fetch] {len(raw_news)} articles for {ticker}")
    return {"raw_news": raw_news}

# Step 3: prompt chain  clean -> classify -> extract -> summarize
# Each LLM step receives the previous step's output, and the whole batch of
# articles goes in one call per step (3 Gemini calls total, not 3 per article).
import re
from typing import Literal

CATEGORIES = ("earnings", "news", "market")
CHAIN_SYSTEM = "You are a careful equity research analyst. Use only the information provided."

class Classified(BaseModel):
    id: int
    category: Literal["earnings", "news", "market"]
    reason: str

class Classification(BaseModel):
    items: List[Classified]

class Extracted(BaseModel):
    id: int
    sentiment: Literal["positive", "negative", "neutral"]
    key_facts: List[str]
    impact: str

class Extraction(BaseModel):
    items: List[Extracted]

class Summary(BaseModel):
    earnings: str
    news: str
    market: str

def ask(prompt: str, schema=None, system: str = CHAIN_SYSTEM):
    """One Gemini call. With `schema`, returns an instance of it; without, returns plain text."""
    if schema is None:
        config = types.GenerateContentConfig(system_instruction=system)
    else:
        config = types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=schema,
        )
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    if schema is None:
        if not response.text:
            raise RuntimeError("Text step returned an empty response")
        return response.text.strip()
    if response.parsed is None:
        raise RuntimeError(f"{schema.__name__} step returned invalid output: {response.text}")
    return response.parsed

def clean_text(text: str, max_chars: int = 1500) -> str:
    text = re.sub(r"<[^>]+>", " ", text or "")   # strip leftover HTML tags
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace
    return text[:max_chars]

def chain(state: State) -> dict:
    ticker = state["ticker"]
    routed = {c: {"items": [], "summary": ""} for c in CATEGORIES}

    # 1. clean (plain Python, no LLM needed)
    articles = []
    for n in state["raw_news"]:
        title, text = clean_text(n.get("title"), 300), clean_text(n.get("text"))
        if title or text:
            articles.append({"id": len(articles), "title": title, "text": text,
                             "source": n.get("source", ""), "published_at": n.get("published_at", ""),
                             "url": n.get("url", "")})
    if not articles:
        print("[chain] no articles to process")
        return {"routed": routed}

    listing = "\n\n".join(f"[{a['id']}] {a['title']}\n{a['text']}" for a in articles)

    # 2. classify
    classification = ask(
        f"Classify each article about {ticker} into exactly one category:\n"
        "- earnings: the company's financial results, guidance, EPS, revenue\n"
        "- news: company-specific events (products, deals, legal/regulatory, analyst ratings, supply chain)\n"
        "- market: macro or sector-wide moves (interest rates, the Fed, indices, sector rotation)\n\n"
        f"Articles:\n{listing}",
        Classification,
    )
    category_of = {c.id: c.category for c in classification.items}

    # 3. extract (uses the categories from step 2)
    labeled = "\n\n".join(
        f"[{a['id']}] ({category_of.get(a['id'], 'news')}) {a['title']}\n{a['text']}" for a in articles
    )
    extraction = ask(
        f"For each article, extract: the sentiment for {ticker} stock, 1-3 key facts "
        f"(keep any numbers), and a one-sentence likely impact on {ticker}'s stock price.\n\n"
        f"Articles:\n{labeled}",
        Extraction,
    )
    details = {e.id: e for e in extraction.items}

    for a in articles:
        cat = category_of.get(a["id"], "news")   # default if the model skipped an id
        e = details.get(a["id"])
        a.update(category=cat,
                 sentiment=e.sentiment if e else "neutral",
                 key_facts=e.key_facts if e else [],
                 impact=e.impact if e else "")
        routed[cat]["items"].append(a)

    # 4. summarize each category (uses the extracted facts from step 3, not raw text)
    facts = "\n\n".join(
        f"{cat.upper()}:\n" + ("\n".join(
            f"- [{a['sentiment']}] {'; '.join(a['key_facts'])} -> {a['impact']}"
            for a in routed[cat]["items"]) or "- (no articles)")
        for cat in CATEGORIES
    )
    summary = ask(
        f"Using only these extracted facts about {ticker}, write a 2-3 sentence summary "
        "for each category. For a category with no articles, write 'No relevant articles.'\n\n"
        f"{facts}",
        Summary,
    )
    for cat in CATEGORIES:
        routed[cat]["summary"] = getattr(summary, cat)

    print("[chain] " + ", ".join(f"{c}: {len(routed[c]['items'])}" for c in CATEGORIES))
    return {"routed": routed}

# Step 4: routing -- send the stock to ONE specialist analyst
def pick_route(routed: dict) -> str:
    if routed["earnings"]["items"]:
        return "earnings"   # earnings move prices the most, so they take priority
    def strength(cat):
        items = routed[cat]["items"]
        pos = sum(a["sentiment"] == "positive" for a in items)
        neg = sum(a["sentiment"] == "negative" for a in items)
        return (len(items), abs(pos - neg))   # more articles first, then more one-sided sentiment
    return max(("news", "market"), key=strength)

def router(state: State) -> str:
    route = pick_route(state["routed"])
    print(f"[router] -> {route}")
    return route

# Step 5: specialist analysts -- same structure, different expertise
ANALYST_FOCUS = {
    "earnings": "You are an earnings analyst. Focus on results versus expectations: EPS and revenue "
                "beats or misses, guidance changes, margins, and what they imply for valuation.",
    "news":     "You are an equity news analyst. Focus on company-specific events: products, partnerships, "
                "legal and regulatory issues, analyst rating changes, and supply chain, and how each "
                "changes the investment case.",
    "market":   "You are a macro and market strategist. Focus on interest rates, Fed policy, sector and "
                "index moves, and how sensitive this stock is to them.",
}

def make_analyst(category: str):
    def analyst(state: State) -> dict:
        ticker, routed = state["ticker"], state["routed"]
        overview = "\n".join(
            f"- {c}: {routed[c]['summary'] or 'No relevant articles.'}" for c in CATEGORIES
        )
        details = "\n".join(
            f"- [{a['sentiment']}] {a['title']} ({a['source']}, {a['published_at'][:10]})\n"
            f"  facts: {'; '.join(a['key_facts'])}\n"
            f"  impact: {a['impact']}"
            for a in routed[category]["items"]
        ) or "- (no articles)"

        prompt = (
            f"Write an investment analysis of {ticker} focused on the {category} evidence.\n\n"
            f"Overview of all evidence:\n{overview}\n\n"
            f"{category.capitalize()} articles in detail:\n{details}\n\n"
            "Use these sections: Thesis (1-2 sentences), Key drivers, Risks, "
            "Outlook (bullish / neutral / bearish, with a one-sentence reason). "
            "Cite the facts above and do not invent numbers."
        )

        # cross-run memory: earlier analyses of this ticker and what the reviewer flagged
        past = load_memory(ticker, k=2)
        if past:
            history = "\n\n".join(
                f"- {r['saved_at'][:10]} ({r['route']} analyst, score {r['score']:.2f}"
                f"{'' if r['accepted'] else ', not accepted'}):\n"
                f"  conclusion (excerpt): {r['analysis'][:600]}\n"
                f"  reviewer feedback: {r['last_critique'][:400]}"
                for r in past
            )
            prompt += (
                f"\n\nYour earlier analyses of {ticker}, newest first:\n{history}\n\n"
                "Briefly state what has changed since the most recent one, and avoid the weaknesses "
                "the reviewer pointed out before. Use these only for comparison: every fact and "
                "number in your analysis must come from today's evidence above."
            )

        # evaluator-optimizer hook: on a retry, revise using the critic's feedback
        if state.get("critique"):
            prompt += (
                f"\n\nRevise your previous analysis to address this reviewer feedback:\n{state['critique']}"
                f"\n\nPrevious analysis:\n{state['analysis']}"
            )

        analysis = ask(prompt, system=ANALYST_FOCUS[category])
        print(f"[{category}_analyst] wrote {len(analysis.split())} words "
              f"(used {len(past)} past analyses)")
        return {"analysis": analysis}

    analyst.__name__ = f"{category}_analyst"
    return analyst

earnings_analyst = make_analyst("earnings")
news_analyst = make_analyst("news")
market_analyst = make_analyst("market")

# Step 6: evaluator-optimizer -- a separate critic scores the analysis;
# a low score sends it back to the same analyst with feedback.
from pydantic import Field

PASS_SCORE = 0.75     # 0-1; at or above this the analysis is accepted
MAX_REVISIONS = 2     # analyst runs at most 1 + MAX_REVISIONS times

CRITIC_SYSTEM = (
    "You are a strict senior investment reviewer. Judge the analysis only against the "
    "evidence given. Do not reward length or confident tone."
)

class Critique(BaseModel):
    grounded: int = Field(ge=0, le=5, description="Every claim and number is supported by the evidence")
    risks: int = Field(ge=0, le=5, description="Material downside risks are identified")
    reasoning: int = Field(ge=0, le=5, description="The outlook follows logically from the drivers and risks")
    clarity: int = Field(ge=0, le=5, description="Specific, concise, and uses the required sections")
    issues: List[str] = Field(description="Concrete problems found; empty if none")
    feedback: str = Field(description="Actionable instructions for the revision")

def critique(state: State) -> dict:
    ticker, routed = state["ticker"], state["routed"]
    evidence = "\n".join(
        f"- ({cat}) [{a['sentiment']}] {a['title']}: {'; '.join(a['key_facts'])}"
        for cat in CATEGORIES for a in routed[cat]["items"]
    ) or "- (no articles)"

    review = ask(
        f"Evaluate this investment analysis of {ticker}.\n\n"
        f"Evidence available to the analyst:\n{evidence}\n\n"
        f"Analysis:\n{state['analysis']}\n\n"
        "Score each criterion from 0 (very poor) to 5 (excellent). List any unsupported or "
        "invented claims under issues, and give specific revision instructions as feedback.",
        Critique,
        system=CRITIC_SYSTEM,
    )
    parts = [review.grounded, review.risks, review.reasoning, review.clarity]
    score = round(sum(min(max(x, 0), 5) for x in parts) / (5 * len(parts)), 2)
    iterations = state.get("iterations", 0) + 1

    feedback = review.feedback
    if review.issues:
        feedback = "Issues:\n" + "\n".join(f"- {i}" for i in review.issues) + f"\n\n{feedback}"

    print(f"[critique] round {iterations}: score {score:.2f} "
          f"(grounded {review.grounded}, risks {review.risks}, "
          f"reasoning {review.reasoning}, clarity {review.clarity})")
    return {"critique": feedback, "score": score, "iterations": iterations}

def after_critique(state: State) -> str:
    """Accept the analysis, or send it back to the analyst that wrote it."""
    if state["score"] >= PASS_SCORE:
        print("[critique] accepted")
        return "done"
    if state["iterations"] > MAX_REVISIONS:
        print(f"[critique] stopping after {MAX_REVISIONS} revisions (best effort)")
        return "done"
    print("[critique] revision requested")
    return router(state)   # routed data is unchanged, so this picks the same analyst

# Step 7: memory -- persist each finished analysis so later runs can look it up
import json
from pathlib import Path

MEMORY_PATH = Path("memory/analyses.jsonl")   # one JSON record per line

def save_memory(state: State) -> dict:
    record = {
        "ticker": state["ticker"],
        "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "route": pick_route(state["routed"]),
        "score": state["score"],
        "accepted": state["score"] >= PASS_SCORE,
        "review_rounds": state["iterations"],
        "headlines": [a["title"] for cat in CATEGORIES for a in state["routed"][cat]["items"]],
        "analysis": state["analysis"],
        "last_critique": state["critique"],
    }
    MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with MEMORY_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"[memory] saved {record['ticker']} analysis (score {record['score']:.2f}) to {MEMORY_PATH}")
    return {}   # writes to disk only; no State fields change

def load_memory(ticker: str, k: int = 3) -> list[dict]:
    """Return the k most recent saved analyses for `ticker`, newest first."""
    if not MEMORY_PATH.exists():
        return []
    records = []
    for line in MEMORY_PATH.read_text(encoding="utf-8").splitlines():
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue   # skip a damaged line instead of failing
        if rec.get("ticker") == ticker.upper():
            records.append(rec)
    return records[::-1][:k]

In [47]:
g = StateGraph(State)

g.add_node("planner", planner)
g.add_node("fetch", fetch)
g.add_node("chain", chain)
g.add_node("earnings", earnings_analyst)
g.add_node("news", news_analyst)
g.add_node("market", market_analyst)
g.add_node("critique", critique)
g.add_node("memory", save_memory)

g.set_entry_point("planner")
g.add_edge("planner", "fetch")
g.add_edge("fetch", "chain")
g.add_conditional_edges("chain", router,
                        {"earnings": "earnings", "news": "news", "market": "market"})
for cat in ("earnings", "news", "market"):
    g.add_edge(cat, "critique")
g.add_conditional_edges("critique", after_critique,
                        {"earnings": "earnings", "news": "news", "market": "market",
                         "done": "memory"})
g.add_edge("memory", END)


app = g.compile()
out = app.invoke({"ticker": "aapl"})
for i, step in enumerate(out["plan"], 1):
    print(i, step)

print()
for n in out["raw_news"][:3]:
    print(n["published_at"], "|", n["source"], "|", n["title"])

for cat, block in out["routed"].items():
    print(f"\n== {cat} ({len(block['items'])} articles) ==")
    print(block["summary"])
    for a in block["items"]:
        print(f"  [{a['sentiment']}] {a['title']}")

print("\n===== ANALYSIS =====")
print(out["analysis"])

print(f"\nscore: {out['score']:.2f} after {out['iterations']} review round(s)")
print("last critique:\n" + out["critique"])

print(f"\nsaved analyses for {out['ticker']}:")
for rec in load_memory(out["ticker"]):
    print(f"  {rec['saved_at']}  route={rec['route']}  score={rec['score']:.2f}")

[fetch] 6 articles for AAPL
[chain] earnings: 1, news: 4, market: 1
[router] -> earnings
[earnings_analyst] wrote 255 words (used 0 past analyses)
[critique] round 1: score 0.95 (grounded 4, risks 5, reasoning 5, clarity 5)
[critique] accepted
[memory] saved AAPL analysis (score 0.95) to memory/analyses.jsonl
1 Gather AAPL historical price action, technical indicators, and valuation multiples across peer benchmarks.
2 Retrieve recent quarterly earnings reports, revenue segment breakdowns, and management guidance updates.
3 Aggregate recent news sentiment, product cycle updates, supply chain reports, and regulatory developments.
4 Perform quantitative and qualitative analysis on cash flow durability, margins, and competitive moat strength.
5 Synthesize findings into an intrinsic valuation model and establish bull, base, and bear price targets.
6 Execute a self-critique by testing alternative bear hypotheses, auditing data assumptions, and identifying blind spots.

2026-10-09T02:04:26+00